# ChessPublishing Atom Extraction — Working Notebook

Iterate on extraction prompt + JSONL format using real examples from the generated data.

**Input:** `data/chesspublishinga.jsonl`

---
## Setup & Data Loading

In [63]:
import os
import json
import random
import chess
import chess.svg
import openai
from IPython.display import display, SVG, Markdown
from dotenv import load_dotenv

load_dotenv()

DATA_FILE = 'data/chesspublishinga.jsonl'

def parse_json_output(text):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('\n', 1)[1]
        if text.endswith('```'):
            text = text[:-3]
        text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"include": False, "exclude_reason": "JSON parse error", "raw": text}

print(f'Data: {DATA_FILE}')

Data: data/chesspublishinga.jsonl


In [64]:
with open(DATA_FILE) as f:
    all_entries = [json.loads(line) for line in f]

# Filter out game-level comments (no move) — we only extract atoms for moves
all_entries = [e for e in all_entries if e.get('move_uci')]

print(f'Loaded {len(all_entries)} entries from {DATA_FILE}')

Loaded 194434 entries from data/chesspublishinga.jsonl


---
## Extraction Prompt & Demos

System prompt, few-shot demos, and helper functions for the extraction LLM call.

In [ ]:
SYSTEM_PROMPT = """\
You are extracting structured move explanations from chess commentary.

You will receive: a FEN, the move played, previous moves for context,
a context_fen, and the annotator's commentary.

For VARIATION entries (sidelines), you will also receive:
- mainline_move: the move the game actually played (this entry discusses
  an alternative)
- parent_comment: framing context from the annotator on the parent move

For MAINLINE entries with sidelines, you will also receive:
- alternatives: structured list of alternative moves from the PGN, each
  with move_san, annotation (the annotator's comment), and line
  (continuation moves). Use these to populate the "alternative" field
  in your output — extract reasoning atoms from the annotator's comments
  on each alternative.

Your job is to EXTRACT and STRUCTURE the commentary into a JSON object.

Output ONLY valid JSON. No preamble, no markdown fences, no explanation.

When include=true, output:
{
  "include": true,
  "fen": "<FEN of the position>",
  "move_san": "<the move>",
  "move_uci": "<UCI of that move>",
  "book_commentary": "<the full original commentary>",
  "reasoning": ["atomic fact 1", "atomic fact 2"],
  "variation": "19...Bxf6 gxf6 20.Rg4+ ...",
  "alternative": {
    "move": "Qxh7+",
    "reasoning": ["why it's worse/better"],
    "variation": "18.Qxh7+ Kf8 ..."
  }
}

When include=false, output ONLY:
{
  "include": false,
  "exclude_reason": "short explanation",
  "book_commentary": "<the full original commentary>"
}

MOVE ATTRIBUTION:
The commentary may explain a PREVIOUS move, not the tagged move.
Use the prev_moves + context_fen to determine which move the commentary
is actually about. The output fen/move_san/move_uci MUST correspond to
the position and move the commentary explains.

ATOM RULES:

1. Each reasoning atom explains something about the CURRENT MOVE from
   the CURRENT FEN. A model will be given only the FEN + move and must
   generate these atoms — so they must make sense in that context alone.

2. STRICT EXTRACTION. Extract ONLY what the commentary explicitly says.
   Do NOT invent, infer, or enrich with your own chess analysis.
   If the commentary says "well timed" but doesn't explain why, that
   is NOT enough — exclude it. The atoms must come from the text.

3. Each atom must explain WHAT the move DOES — a threat it creates, a
   square it controls, a piece it activates, a plan it enables.
   GOOD: "Qd4 prevents Black's rooks from reaching the central files."
   GOOD: "d5 prevents White from playing e4."
   BAD:  "Nd4 is a strong resource for White." (vague label)
   BAD:  "Rg8 allows Black to hold on." (doesn't say what Rg8 does)
   BAD:  "Be5 is presented as a standard idea." (meta-description)

4. Short illustrative lines in atoms are fine to show a threat or idea
   (e.g. "Bxd4 threatens Bxf6 gxf6, Rg4+ with a mating attack").
   Long multi-move variations (5+ moves) belong in the "variation" field,
   NOT inside atoms.

5. Do NOT prefix atoms with preceding moves from earlier in the game.
   The atom must be about the position in the FEN, not how we got there.

6. Group logically connected setup+consequence into a SINGLE atom.

7. Name squares, pieces, and diagonals concretely.

8. DROP VAGUE LABELS. Silently drop phrases that are merely evaluative
   labels — even when the entry has other good atoms. Examples to drop:
   "The move is committal", "A slightly passive choice",
   "This keeps the tension", "A practical decision",
   "A strong resource", "Other moves bring no problems".

9. INCLUDE/EXCLUDE.
   EXCLUDE when:
   - The commentary contains NO concrete facts about what the move does
     (only labels like "well timed", "natural", "a good move")
   - Commentary only shows a variation line with no explanation of WHY
     the move works (a bare line + "holds"/"nothing clear"/"wins" is
     NOT an explanation)
   - Historical anecdotes, biographies, game references with no analysis
   INCLUDE when there is at least one concrete fact about what the move
   does — a specific square, piece, threat, file, diagonal, or plan.
   "(stopping e4)" is enough. "well timed" is not.

10. NEVER include generic philosophical statements as atoms.

11. For sidelines: the alternative field captures the MAINLINE move that
    the annotator is comparing against, if they discuss it.

12. For mainline entries with "alternatives" provided: use the annotator's
    comments and lines from the alternatives to populate the "alternative"
    field. Each alternative may have annotation text and a continuation
    line — extract reasoning atoms from these just as you would from the
    main commentary. If there are multiple alternatives, output
    "alternative" as a list of objects.
"""

print(f'System prompt: {len(SYSTEM_PROMPT)} chars')

In [66]:
# ── Hardcoded demo data (from Chernev's Logical Chess) ──────────────────
# These are 3 Chernev demos for few-shot extraction. No engine data needed.

DEMO_DATA = [
    # Demo 1: Qe5 — tactical (pin + attack + rejected alternative)
    {
        'fen': 'rn2k1nr/ppp2ppp/8/q7/1b1N2b1/2N5/PPPBBPPP/R2QK2R b KQkq - 0 8',
        'move_uci': 'a5e5',
        'move_san': 'Qe5',
        'annotation': (
            'Black\'s response pins the e2-bishop and attacks the unprotected '
            'd4-knight. Black rejects 8... Bxe2 as the recapture by 9 Qxe2+ '
            'gains another tempo for White.'),
        'prev_moves': '',
        'context_fen': '',
        'is_mainline': True,
        'mainline_move': None,
        'parent_comment': None,
    },
    # Demo 2: Bxd4 — variation with mating attack
    {
        'fen': 'r4rk1/pp1q1ppp/4pb2/8/2PpR3/1P2Q3/PB3PPP/5RK1 w - - 0 19',
        'move_uci': 'b2d4',
        'move_san': 'Bxd4',
        'annotation': (
            'White regains the pawn, and his bishop now attacks in two directions. '
            'On the one hand, it threatens to take the a-pawn, on the other it aims '
            'at checkmate by... Bxf6 gxf6 21 Rg4+ Kh8 22 Qh6 Rg8 23 Qxf6+ and '
            'mate next move.'
        ),
        'prev_moves': '',
        'context_fen': '',
        'is_mainline': True,
        'mainline_move': None,
        'parent_comment': None,
    },
    # Demo 3: O-O — mistake with detailed alternative
    {
        'fen': 'r2qk2r/p1p1npp1/1pn1b2p/3pP3/3P1B2/2PB1N2/P1PQ2PP/1R3RK1 b kq - 1 12',
        'move_uci': 'e8g8',
        'move_san': 'O-O',
        'annotation': (
            'Walking right into the teeth of the storm!\n'
            'Before making a move that suggests itself so readily, Black might have '
            'asked himself, "How can I exploit White\'s one weakness, the doubled '
            'pawns on the c-file?"\n'
            'He might then have hit upon 12... Na5, with the object of swinging the '
            'knight to c4. There it blockades the doubled pawn, interferes with the '
            'free movement of White\'s pieces, and in general sticks like a bone in '
            'the throat. White could capture the knight, but then he parts with one '
            'of his valuable bishops, and as a result of the exchange his pawn '
            'position would be inferior to Black\'s. Finally, Black could then anchor '
            'one of his pieces to great effect on d5, a square from which it could '
            'never be evicted by pawns.'
        ),
        'prev_moves': '',
        'context_fen': '',
        'is_mainline': True,
        'mainline_move': None,
        'parent_comment': None,
    },
]

DEMO_FENS = {d['fen'] for d in DEMO_DATA}

# ── Demo responses ───────────────────────────────────────────────────────
# Atoms explain WHY from the current position — no prev-move prefixes,
# no long variation dumps. Short illustrative lines are OK.

_DEMO_RESPONSES = [
    # Demo 1: Qe5 (tactical — pin + attack + rejected line)
    {
        'include': True,
        'fen': 'rn2k1nr/ppp2ppp/8/q7/1b1N2b1/2N5/PPPBBPPP/R2QK2R b KQkq - 0 8',
        'move_san': 'Qe5',
        'move_uci': 'a5e5',
        'reasoning': [
            'Qe5 pins the bishop on e2 to the king on e1, since the queen on e5 attacks along the e-file.',
            'Qe5 simultaneously attacks the unprotected knight on d4.',
            'Black rejects Bxe2 because after Qxe2+, White recaptures with check, gaining a tempo.',
        ],
    },
    # Demo 2: Bxd4 (mating attack — short threat in atom, full line in variation)
    {
        'include': True,
        'fen': 'r4rk1/pp1q1ppp/4pb2/8/2PpR3/1P2Q3/PB3PPP/5RK1 w - - 0 19',
        'move_san': 'Bxd4',
        'move_uci': 'b2d4',
        'reasoning': [
            'Bxd4 recaptures the pawn on d4.',
            'The bishop on d4 threatens to capture the undefended a7-pawn.',
            'The bishop on d4 also enables a mating threat: Bxf6 gxf6, Rg4+ Kh8, Qh6 with Qxf6+ and mate to follow.',
        ],
        'variation': 'Bxf6 gxf6 Rg4+ Kh8 Qh6 Rg8 Qxf6+',
    },
    # Demo 3: O-O (mistake — castles into attack; Na5 alternative with plans)
    {
        'include': True,
        'fen': 'r2qk2r/p1p1npp1/1pn1b2p/3pP3/3P1B2/2PB1N2/P1PQ2PP/1R3RK1 b kq - 1 12',
        'move_san': 'O-O',
        'move_uci': 'e8g8',
        'reasoning': [
            "O-O castles into White's prepared kingside attack.",
            "Black misses the opportunity to exploit White's doubled c-pawns.",
        ],
        'alternative': {
            'move': 'Na5',
            'reasoning': [
                'Na5 reroutes the knight toward c4 to blockade the doubled c-pawn.',
                'A knight on c4 would interfere with the coordination of White\'s pieces.',
                'If White captures Bxc4 dxc4, the resulting pawn structure favors Black.',
                'After Na5, Black can later anchor a piece on d5, a square no pawn can attack.',
            ],
        },
    },
]

print(f'Demo data: {len(DEMO_DATA)} positions, {len(DEMO_FENS)} FENs')

Demo data: 3 positions, 3 FENs


In [ ]:
def build_user_prompt(entry):
    """Build user prompt: FEN + move + prev_moves + context_fen + mainline_move + parent_comment + alternatives + commentary + line."""
    board = chess.Board(entry['fen'])
    move_san = entry.get('move_san') or board.san(chess.Move.from_uci(entry['move_uci']))
    turn = 'White' if board.turn == chess.WHITE else 'Black'

    prompt = f"FEN: {entry['fen']}\n"
    prompt += f"Move played: {move_san} ({turn})\n"

    if entry.get('prev_moves'):
        prompt += f"Previous moves: {entry['prev_moves']}\n"
    if entry.get('context_fen'):
        prompt += f"Context FEN (start of prev_moves): {entry['context_fen']}\n"

    # Variation context: what the mainline played + parent framing
    if entry.get('mainline_move'):
        prompt += f"Mainline move: {entry['mainline_move']} (this entry discusses {move_san} as an alternative)\n"
    if entry.get('parent_comment'):
        prompt += f"Parent context: \"{entry['parent_comment']}\"\n"

    # Alternatives from PGN sibling variations
    if entry.get('alternatives'):
        prompt += "\nAlternative variations from the PGN:\n"
        for alt in entry['alternatives']:
            prompt += f"  - {alt['move_san']}"
            if alt.get('annotation'):
                prompt += f": {alt['annotation']}"
            if alt.get('line'):
                prompt += f" (line: {alt['line']})"
            prompt += "\n"

    prompt += f'\nCommentary: \"{entry["annotation"]}\"\n'
    if entry.get('line'):
        prompt += f"Continuation line: {entry['line']}\n"
    return prompt


def postprocess_filter(parsed_json):
    """Post-processing filters that catch what the LLM might miss."""
    if parsed_json.get('include', True) and not parsed_json.get('reasoning'):
        return False, "no reasoning atoms extracted"
    alt = parsed_json.get('alternative')
    if alt:
        alts = alt if isinstance(alt, list) else [alt]
        if any(not a.get('reasoning') for a in alts):
            return False, "alternative has no reasoning — likely not from commentary"
    return parsed_json.get('include', True), parsed_json.get('exclude_reason')

print('Helpers ready')

In [68]:
def build_demos(demo_data, demo_responses):
    """Build few-shot demo messages from hardcoded data + responses."""
    demos = []
    for data, resp in zip(demo_data, demo_responses):
        r = {**resp, 'book_commentary': data['annotation']}
        demos.append({'role': 'user', 'content': build_user_prompt(data)})
        demos.append({'role': 'assistant', 'content': json.dumps(r, indent=2)})
    return demos


DEMOS = build_demos(DEMO_DATA, _DEMO_RESPONSES)

print(f'Demos: {len(DEMOS) // 2} examples')

# Show what goes to the LLM
for i in range(0, len(DEMOS), 2):
    n = i // 2 + 1
    print(f'\n{"="*60}')
    print(f'Demo {n} — USER:')
    print(DEMOS[i]['content'][:200] + '...')
    print(f'\nDemo {n} — ASSISTANT:')
    print(DEMOS[i+1]['content'])

Demos: 3 examples

Demo 1 — USER:
FEN: rn2k1nr/ppp2ppp/8/q7/1b1N2b1/2N5/PPPBBPPP/R2QK2R b KQkq - 0 8
Move played: Qe5 (Black)

Commentary: "Black's response pins the e2-bishop and attacks the unprotected d4-knight. Black rejects 8... ...

Demo 1 — ASSISTANT:
{
  "include": true,
  "fen": "rn2k1nr/ppp2ppp/8/q7/1b1N2b1/2N5/PPPBBPPP/R2QK2R b KQkq - 0 8",
  "move_san": "Qe5",
  "move_uci": "a5e5",
  "reasoning": [
    "Qe5 pins the bishop on e2 to the king on e1, since the queen on e5 attacks along the e-file.",
    "Qe5 simultaneously attacks the unprotected knight on d4.",
    "Black rejects Bxe2 because after Qxe2+, White recaptures with check, gaining a tempo."
  ],
  "book_commentary": "Black's response pins the e2-bishop and attacks the unprotected d4-knight. Black rejects 8... Bxe2 as the recapture by 9 Qxe2+ gains another tempo for White."
}

Demo 2 — USER:
FEN: r4rk1/pp1q1ppp/4pb2/8/2PpR3/1P2Q3/PB3PPP/5RK1 w - - 0 19
Move played: Bxd4 (White)

Commentary: "White regains the pawn, 

In [69]:
MODEL = "gpt-5.4"

client = openai.OpenAI()

def call_llm(messages, model=MODEL, temperature=0.3):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=2048,
    )
    return resp.choices[0].message.content

def generate_extraction(entry, model=MODEL):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(DEMOS)
    messages.append({"role": "user", "content": build_user_prompt(entry)})
    return call_llm(messages, model=model)

print(f'Model: {MODEL}')

Model: gpt-5.4


---
## Run Extraction & Review

For each sampled position: get engine analysis, run extraction, display board + commentary + atoms.

In [ ]:
# ── Config (change these and re-run just this cell) ──
SEED = 42
NUM_POSITIONS = 10

rng = random.Random(SEED)
sample_indices = rng.sample(range(len(all_entries)), NUM_POSITIONS)
samples = [all_entries[i] for i in sorted(sample_indices)]

print(f'Sampled {NUM_POSITIONS} positions (seed={SEED})')
for i, e in enumerate(samples):
    tag = 'ML' if e['is_mainline'] else f'V(d={e["variation_depth"]})'
    ml = f' [alt to {e["mainline_move"]}]' if e.get('mainline_move') else ''
    print(f'  [{i}] {tag}{ml} {e["game"]} — {e["move_san"]}  ({len(e["annotation"])} chars)')

for idx, entry in enumerate(samples):
    board = chess.Board(entry['fen'])
    move = chess.Move.from_uci(entry['move_uci'])
    move_san = board.san(move)
    turn = 'White' if board.turn == chess.WHITE else 'Black'

    # Run extraction
    raw_text = generate_extraction(entry)
    parsed = parse_json_output(raw_text)

    # Post-process filter
    pp_include, pp_reason = postprocess_filter(parsed)
    if not pp_include and parsed.get('include', True):
        parsed['include'] = False
        parsed['exclude_reason'] = pp_reason

    # ── Display ──
    svg = chess.svg.board(board, arrows=[(move.from_square, move.to_square)], size=300)
    display(SVG(svg))

    meta = entry.get('metadata', {})
    game = meta.get('White', '?') + ' vs ' + meta.get('Black', '?')

    # Title with variation info
    title = f'### [{idx+1}/{len(samples)}] {game} — {turn} plays {move_san}'
    if entry.get('mainline_move'):
        title += f' (alternative to {entry["mainline_move"]})'
    display(Markdown(title))

    if entry.get('prev_moves'):
        display(Markdown(f'**Prev moves:** `{entry["prev_moves"]}`'))

    if entry.get('parent_comment'):
        display(Markdown(f'**Parent context:** *{entry["parent_comment"]}*'))

    # Show PGN alternatives (input context)
    if entry.get('alternatives'):
        alt_lines = []
        for a in entry['alternatives']:
            line = f'  - **{a["move_san"]}**'
            if a.get('annotation'):
                line += f': {a["annotation"]}'
            if a.get('line'):
                line += f' *(line: {a["line"]})*'
            alt_lines.append(line)
        display(Markdown('**PGN alternatives:**\n' + '\n'.join(alt_lines)))

    display(Markdown(f'> {entry["annotation"]}'))

    if entry.get('line'):
        display(Markdown(f'**Continuation line:** `{entry["line"]}`'))

    # Extraction result
    included = parsed.get('include', False)
    tag = 'INCLUDED' if included else f'EXCLUDED: {parsed.get("exclude_reason", "?")}'
    display(Markdown(f'#### Extraction [{tag}]'))

    reasoning = parsed.get('reasoning', [])
    if reasoning:
        display(Markdown('\n'.join(f'{i+1}. {a}' for i, a in enumerate(reasoning))))

    alt = parsed.get('alternative')
    if alt:
        alts = alt if isinstance(alt, list) else [alt]
        for a in alts:
            r_list = a.get('reasoning', [])
            display(Markdown(
                f'**Alternative: {a.get("move", "?")}**\n' +
                '\n'.join(f'  - {r}' for r in r_list)
            ))

    if parsed.get('variation'):
        display(Markdown(f'**Variation:** `{parsed["variation"]}`'))

    # Full structured JSON
    display(Markdown(f'<details><summary>Full JSON</summary>\n\n```json\n{json.dumps(parsed, indent=2)}\n```\n</details>'))

    display(Markdown('---'))

In [ ]:
# ── Config (change these and re-run just this cell) ──
SEED = 43
NUM_POSITIONS = 10

rng = random.Random(SEED)
sample_indices = rng.sample(range(len(all_entries)), NUM_POSITIONS)
samples = [all_entries[i] for i in sorted(sample_indices)]

print(f'Sampled {NUM_POSITIONS} positions (seed={SEED})')
for i, e in enumerate(samples):
    tag = 'ML' if e['is_mainline'] else f'V(d={e["variation_depth"]})'
    ml = f' [alt to {e["mainline_move"]}]' if e.get('mainline_move') else ''
    print(f'  [{i}] {tag}{ml} {e["game"]} — {e["move_san"]}  ({len(e["annotation"])} chars)')

for idx, entry in enumerate(samples):
    board = chess.Board(entry['fen'])
    move = chess.Move.from_uci(entry['move_uci'])
    move_san = board.san(move)
    turn = 'White' if board.turn == chess.WHITE else 'Black'

    # Run extraction
    raw_text = generate_extraction(entry)
    parsed = parse_json_output(raw_text)

    # Post-process filter
    pp_include, pp_reason = postprocess_filter(parsed)
    if not pp_include and parsed.get('include', True):
        parsed['include'] = False
        parsed['exclude_reason'] = pp_reason

    # ── Display ──
    svg = chess.svg.board(board, arrows=[(move.from_square, move.to_square)], size=300)
    display(SVG(svg))

    meta = entry.get('metadata', {})
    game = meta.get('White', '?') + ' vs ' + meta.get('Black', '?')

    # Title with variation info
    title = f'### [{idx+1}/{len(samples)}] {game} — {turn} plays {move_san}'
    if entry.get('mainline_move'):
        title += f' (alternative to {entry["mainline_move"]})'
    display(Markdown(title))

    if entry.get('prev_moves'):
        display(Markdown(f'**Prev moves:** `{entry["prev_moves"]}`'))

    if entry.get('parent_comment'):
        display(Markdown(f'**Parent context:** *{entry["parent_comment"]}*'))

    # Show PGN alternatives (input context)
    if entry.get('alternatives'):
        alt_lines = []
        for a in entry['alternatives']:
            line = f'  - **{a["move_san"]}**'
            if a.get('annotation'):
                line += f': {a["annotation"]}'
            if a.get('line'):
                line += f' *(line: {a["line"]})*'
            alt_lines.append(line)
        display(Markdown('**PGN alternatives:**\n' + '\n'.join(alt_lines)))

    display(Markdown(f'> {entry["annotation"]}'))

    if entry.get('line'):
        display(Markdown(f'**Continuation line:** `{entry["line"]}`'))

    # Extraction result
    included = parsed.get('include', False)
    tag = 'INCLUDED' if included else f'EXCLUDED: {parsed.get("exclude_reason", "?")}'
    display(Markdown(f'#### Extraction [{tag}]'))

    reasoning = parsed.get('reasoning', [])
    if reasoning:
        display(Markdown('\n'.join(f'{i+1}. {a}' for i, a in enumerate(reasoning))))

    alt = parsed.get('alternative')
    if alt:
        alts = alt if isinstance(alt, list) else [alt]
        for a in alts:
            r_list = a.get('reasoning', [])
            display(Markdown(
                f'**Alternative: {a.get("move", "?")}**\n' +
                '\n'.join(f'  - {r}' for r in r_list)
            ))

    if parsed.get('variation'):
        display(Markdown(f'**Variation:** `{parsed["variation"]}`'))

    # Full structured JSON
    display(Markdown(f'<details><summary>Full JSON</summary>\n\n```json\n{json.dumps(parsed, indent=2)}\n```\n</details>'))

    display(Markdown('---'))